# Creating simulated datasets

This is step 1 of the ISO end-to-end pipeline:

1. **this notebook** synthesises the datasets,
2. [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb) builds the cross-covariance
   between them,
3. [`analyse_datasets.ipynb`](analyse_datasets.ipynb) runs the joint fit on exactly these products.

All artifacts go to a stable, git-ignored `sims/` directory that the downstream notebooks read by
the same filenames. We build, from simplest to most involved:

1. **From scratch** — a galaxy x CMB-lensing cross-correlation, computed with
   [CCL](https://github.com/LSSTDESC/CCL), with the cosmology and the noise exposed as knobs.
2. **A smooth CMB-lensing twin** — the shipped reconstruction with its bandpowers replaced by the
   theory at the ISO fiducial (chi-square = 0 when fit at the same cosmology), via
   `soliket.sacc_tools.smooth_twin_sacc`.
3. **A smooth MFLike CMB+foregrounds dataset** — the per-frequency theory binned through MFLike's own
   windows, via `soliket.sacc_tools.smooth_mflike_sacc`.

Datasets 2 and 3 are the joint-analysis inputs; everything is built at the **ISO fiducial**
(`defaults_dir="defaults"` — cosmology incl. the SO normal-hierarchy neutrinos).

In [ ]:
import os
import shutil
from pathlib import Path

import numpy as np
import sacc

# All three ISO notebooks share this directory under the same name (git-ignored).
SIMS = Path("sims")
SIMS.mkdir(exist_ok=True)

# Canonical artifact filenames the downstream notebooks read.
GALAXY_KAPPA = SIMS / "galaxy_kappa.sim.fits"
LENSING_SMOOTH = SIMS / "lensing_smooth.sacc.fits"
MFLIKE_SMOOTH = SIMS / "mflike_smooth.fits"
print("writing simulated datasets to", SIMS.resolve())

## 1. A galaxy x kappa dataset from scratch

We simulate the cross-correlation of an unWISE-like galaxy sample with the SO CMB-lensing
convergence. The three spectra (`gg`, `gk`, `kk`) come from CCL; the bandpower windows and the joint
covariance come from `soliket.sacc_tools`. We wrap it in a function so the **cosmology** and the
**galaxy noise spectrum** are explicit knobs, then build the canonical dataset and two variants.

In [ ]:
import pyccl as ccl

from soliket.sacc_tools import gaussian_covariance, top_hat_windows

# Fiducial galaxy redshift distribution, borrowed from the shipped reference dataset.
ref = sacc.Sacc.load_fits("../../tests/data/unwise_g-so_kappa.sim.sacc.fits")
Z, NZ = ref.tracers["gc_unwise"].z, ref.tracers["gc_unwise"].nz


def build_galaxy_kappa(out_path, *, cosmo, ngal_per_arcmin2=1.0, fsky=0.4,
                       ell_max=600, n_bins=20, noise_gg=None):
    """Simulate an unWISE-like galaxy x SO CMB-lensing cross-correlation SACC.

    `cosmo` is a ``ccl.Cosmology``; `noise_gg` optionally overrides the galaxy
    auto-spectrum noise (default: shot noise ``1 / n_gal``). Returns the SACC.
    """
    b1, mag_bias = 1.0, 0.4
    gc = ccl.NumberCountsTracer(
        cosmo, has_rsd=False, dndz=(Z, NZ),
        bias=(Z, b1 * np.ones_like(Z)), mag_bias=(Z, mag_bias * np.ones_like(Z)),
    )
    ck = ccl.CMBLensingTracer(cosmo, z_source=1086.0)

    ells, window = top_hat_windows(ell_max, n_bins)
    delta_ell = ell_max // n_bins
    cl_gg = ccl.angular_cl(cosmo, gc, gc, ells)
    cl_gk = ccl.angular_cl(cosmo, gc, ck, ells)
    cl_kk = ccl.angular_cl(cosmo, ck, ck, ells)

    if noise_gg is None:                       # default: galaxy shot noise 1 / n_gal
        ngal_sr = ngal_per_arcmin2 / np.deg2rad(1.0 / 60.0) ** 2
        noise_gg = np.full_like(cl_gg, 1.0 / ngal_sr)
    cls = np.array([[cl_gg + noise_gg, cl_gk], [cl_gk, cl_kk]])
    cov = gaussian_covariance(cls, ells, delta_ell, fsky)

    s = sacc.Sacc()
    s.metadata["info"] = "Simulated unWISE-like galaxy x SO CMB-lensing cross-correlation"
    s.add_tracer("NZ", "gc_unwise", quantity="galaxy_density", spin=0,
                 z=Z, nz=NZ, metadata={"ngal": ngal_per_arcmin2})
    s.add_tracer("Map", "ck_so", quantity="cmb_convergence", spin=0,
                 ell=np.arange(3000), beam=np.ones(3000))
    s.add_ell_cl("cl_00", "gc_unwise", "gc_unwise", ells, cl_gg, window=window)
    s.add_ell_cl("cl_00", "gc_unwise", "ck_so",     ells, cl_gk, window=window)
    s.add_ell_cl("cl_00", "ck_so",     "ck_so",     ells, cl_kk, window=window)
    s.add_covariance(cov)
    s.save_fits(str(out_path), overwrite=True)
    return s

In [ ]:
# Canonical dataset at the fiducial cosmology.
fiducial = ccl.Cosmology(Omega_c=0.25, Omega_b=0.05, h=0.7, n_s=0.965, A_s=2.11e-9,
                         matter_power_spectrum="linear")
s = build_galaxy_kappa(GALAXY_KAPPA, cosmo=fiducial)
print("wrote", GALAXY_KAPPA.name, "->", len(s.mean), "data points")

# Knob 1 - a different cosmology (lower A_s -> lower sigma8).
low_amp = ccl.Cosmology(Omega_c=0.25, Omega_b=0.05, h=0.7, n_s=0.965, A_s=1.8e-9,
                        matter_power_spectrum="linear")
build_galaxy_kappa(SIMS / "galaxy_kappa.lowA.fits", cosmo=low_amp)

# Knob 2 - a custom (flat, noisier) galaxy noise spectrum instead of pure shot noise.
ells, _ = top_hat_windows(600, 20)
build_galaxy_kappa(SIMS / "galaxy_kappa.noisy.fits", cosmo=fiducial,
                   noise_gg=np.full(len(ells), 5e-6))
print("knob variants written (different cosmology, custom noise)")

## 2. A smooth CMB-lensing twin

A *smooth* twin reuses the shipped dataset's tracers, bandpower windows and covariance but replaces
the noisy measured bandpowers with the theory at a chosen cosmology — so the likelihood gives
chi-square = 0 when fit at that cosmology. `soliket.sacc_tools.smooth_twin_sacc` does the SACC
surgery; we get the binned theory from an evaluated `lensing` likelihood (via `resolve_aliases`, the
named role — not a `Session`). The **imprint cosmology is a knob**: by default the ISO fiducial, but
any param override (e.g. a shifted `tau`) imprints a non-fiducial twin for parameter-recovery tests.

In [ ]:
from cobaya.model import get_model
from cobaya.tools import resolve_packages_path

from soliket.presets import build_info, resolve_aliases
from soliket.sacc_tools import smooth_twin_sacc


def smooth_lensing(out_path, **param_overrides):
    """Write a smooth (theory) CMB-lensing twin at the ISO fiducial.

    `param_overrides` pins fiducial params (e.g. ``tau=0.06``) to imprint the twin
    at a non-fiducial cosmology. Returns ``(lensing_likelihood, binned_clkk)``.
    """
    info = build_info("lensing", defaults_dir="defaults")
    info["packages_path"] = resolve_packages_path()
    for name, value in param_overrides.items():
        info["params"][name] = {"value": value}

    model = get_model(info)
    model.loglikes({})                          # evaluate at the imprint cosmology
    lensing = resolve_aliases(model).lensing
    clkk = lensing._get_theory()                # binned C_ell^kappakappa

    src = sacc.Sacc.load_fits(lensing.datapath)  # reuse shipped tracers/windows/cov
    smooth_twin_sacc(src, "cl_00", "ck", "ck", clkk, out_path=out_path)
    return lensing, clkk


lensing, clkk = smooth_lensing(LENSING_SMOOTH)
print("wrote", LENSING_SMOOTH.name, "->", len(clkk), "bins")

In [ ]:
# Knob - imprint a second twin at a shifted tau (a parameter-recovery target).
smooth_lensing(SIMS / "lensing_smooth.tau0p06.sacc.fits", tau=0.06)
print("wrote a tau=0.06 twin alongside")

## 3. A smooth MFLike CMB + foregrounds dataset

The same smooth-twin idea for the primary CMB, where the data vector spans many frequency
cross-spectra. The per-frequency plumbing — combining CMB + foregrounds + systematics through
MFLike's `get_modified_theory`, binning with MFLike's own bandpower windows, and writing one `NuMap`
tracer per `(frequency, spin)` channel — lives in `soliket.sacc_tools.smooth_mflike_sacc`, which
takes the concrete handles (the evaluated likelihood + theory outputs, not a `Session`).

This build runs CAMB at MFLike accuracy and takes a few minutes; the covariance and bandpower-window
(Bbl) matrices are reused from the shipped `cov_Bbl_file`, so only the data vector is regenerated.

In [ ]:
from soliket.sacc_tools import smooth_mflike_sacc

RUN_MFLIKE = True  # set False to skip the few-minute CAMB build

if RUN_MFLIKE:
    info = build_info("mflike", defaults_dir="defaults")
    info["packages_path"] = resolve_packages_path()
    model = get_model(info)
    roles = resolve_aliases(model)

    # Numeric fiducial values (skip lambda-valued / derived params).
    params = {k: v["value"] for k, v in info["params"].items()
              if isinstance(v, dict) and "value" in v and not isinstance(v["value"], str)}
    model.loglikes(params)                      # evaluate at the ISO fiducial

    dls = model.provider.get_Cl(ell_factor=True)
    fg_totals = roles.foreground.get_fg_totals()
    smooth_mflike_sacc(roles.mflike, dls, fg_totals, params, out_path=MFLIKE_SMOOTH)
    print("wrote", MFLIKE_SMOOTH.name)
else:
    print("RUN_MFLIKE is False - skipping the smooth MFLike build.")

## Recap

`sims/` now holds the pipeline inputs:

| Artifact | Built from | Consumed by |
| --- | --- | --- |
| `mflike_smooth.fits` | `smooth_mflike_sacc` (CMB+fg theory, ISO fiducial) | the analysis (MFLike component) |
| `lensing_smooth.sacc.fits` | `smooth_twin_sacc` (lensing theory, ISO fiducial) | the analysis (lensing component) |
| `galaxy_kappa.sim.fits` (+ knob variants) | CCL from scratch | standalone galaxy x kappa example |

The smooth lensing twin carries only the data + covariance; the analysis notebook points the
likelihood's `correction_filename` / `fiducial_filename` at the shipped (multi-hundred-MB) auxiliary
files in place, so we never copy those into `sims/`. Both smooth twins are built at the **same ISO
fiducial** the analysis fits at, so the joint chi-square is 0 by construction.

Next: [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb) builds the cross-covariance
between the MFLike and lensing data vectors; then
[`analyse_datasets.ipynb`](analyse_datasets.ipynb) runs the joint fit on these `sims/` products.